Data Aggregation Stuff

Kyle - Given a latitude and longitude and a radius and a set of points, find every point in that radius and turn that into the individual datasets

the idea is make some python functions which do this, geopandas might be the right library to use?
this will enable the joining of our many datasets: each row in our final dataset would be e.g. one sensor location with the aggregate stat for each associated dataset within that radius - so sum of building fuel use, sum of construction permits, sum of traffic (you don't have to do the aggregation stuff for now since we'll need to think more about the right way to do it)


In [19]:
# STANDARDIZATION

STANDARD_COLUMNS = [
    "source",
    "sensor_id",
    "latitude",
    "longitude",
    "year",
    "month",
    "pm25"
]

def standardize_sensor_data(df, source, column_map):
    # Rename columns
    result = df.rename(columns=column_map).copy()

    # Add data source
    result["source"] = source

    # 4. Only keep the standardized columns
    result = result[STANDARD_COLUMNS]

    # 5. Standardize data types
    result["sensor_id"] = result["sensor_id"].astype("string")

    for col in ["latitude", "longitude", "pm25"]:
        result[col] = pd.to_numeric(
            result[col], errors="coerce"
        )

    for col in ["year", "month"]:
        result[col] = pd.to_numeric(
            result[col], errors="coerce"
        ).astype("Int64")

    # 6. Remove invalid coordinates
    result = result.dropna(
        subset=["sensor_id", "latitude", "longitude"]
    )

    # 7. Validate coordinates and dates
    valid = (
        result["latitude"].between(-90, 90) &
        result["longitude"].between(-180, 180) &
        result["month"].between(1, 12) &
        result["year"].notna()
    )

    result = result.loc[valid.fillna(False)].copy()

    # 8. Create source-qualified sensor ID
    result["sensor_key"] = (
        result["source"] + "_" + result["sensor_id"]
    )

    return result


epa_col_map =  {
    "Site Num": "sensor_id",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Year": "year",
    "Month": "month",
    "Arithmetic Mean": "pm25"
}
openaq_col_map =  {
    "location_id": "sensor_id",
    "latitude": "latitude",
    "longitude": "longitude",
    "year": "year",
    "month": "month",
    "pm25_mean_ug_m3": "pm25"
}
purple_col_map =  {
    "sensor_index": "sensor_id",
    "latitude": "latitude",
    "longitude": "longitude",
    "Year": "year",
    "Month": "month",
    "pm2.5_atm": "pm25"
}

In [20]:
import pandas as pd
import geopandas as gpd

# Find data for all of our features: greenery, traffic, boiler, building, etc.
# Convert each dataset to a geopandas dataframe
# Spatially join on our sensor data using a radius
# Aggregate the values of the records that are within the sensor's radius
# Now we have a aggregate dataset with sensor --> aggregate levels of our features
# Can create ML model for this!


def to_gdf(df, lat_col="latitude", lon_col="longitude"):
    df = df.dropna(subset=[lat_col, lon_col]).copy()
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326",
    )
    return gdf.to_crs("EPSG:32618")


def aggregate_near_sensors(
    sensors_gdf,
    features_gdf,
    prefix,
    radius=1609.344,
    mode="monthly",
    year_col=None,
    month_col=None,
    agg_cols=None,
    agg_type="sum"
):
    # Ensure same CRS
    features_gdf = features_gdf.to_crs(sensors_gdf.crs)

    # STEP 1: Prepare temporal columns
    if mode == "monthly":
        features_gdf = features_gdf.rename(
            columns={
                year_col: "feature_year",
                month_col: "feature_month"
            }
        )
        time_cols = ["feature_year", "feature_month"]

    elif mode == "yearly":
        features_gdf = features_gdf.rename(
            columns={year_col: "feature_year"}
        )
        time_cols = ["feature_year"]

    elif mode == "interval":
        time_cols = ["issue_ym", "expiration_ym"] # pre-processed to improve speed

    elif mode == "static":
        time_cols = []

    else:
        raise ValueError("Invalid mode")

    # STEP 2: Determine columns needed for aggregation
    if agg_cols is None:
        agg_cols = {}

    join_cols = list(dict.fromkeys(
        [*time_cols, *agg_cols.keys(), "geometry"]
    ))

    # STEP 3: Spatial join
    nearby = gpd.sjoin(
        sensors_gdf,
        features_gdf[join_cols],
        how="inner",
        predicate="dwithin",
        distance=radius
    )

    # STEP 4: Temporal filtering
    if mode == "monthly":
        nearby = nearby[
            (nearby["year"] == nearby["feature_year"]) &
            (nearby["month"] == nearby["feature_month"])
        ]

    elif mode == "yearly":
        nearby = nearby[
            nearby["year"] == nearby["feature_year"]
        ]

    elif mode == "interval":
        nearby = nearby[
            (nearby["sensor_ym"] >= nearby["issue_ym"]) &
            (nearby["sensor_ym"] <= nearby["expiration_ym"])
        ]

    # Static mode: no temporal filtering

    # STEP 5: Aggregate
    group_cols = ["sensor_key", "year", "month"]

    if agg_type == "count":
        aggregated = (
            nearby.groupby(group_cols)
            .size()
            .reset_index(name=f"{prefix}_count")
        )

    elif agg_type in ["sum", "mean"]:
        aggregated = (
            nearby.groupby(group_cols)
            .agg(
                **{
                    f"{prefix}_{col}_{stat}": (col, stat)
                    for col, stat in agg_cols.items()
                }
            )
            .reset_index()
        )

    else:
        raise ValueError("agg_type must be 'sum', 'mean', or 'count'")

    return aggregated

In [21]:
# boiler_gdf.columns

In [22]:
building_data_df = pd.read_csv("Datas/buildings_data.csv")[['Calendar Year', 'Latitude', 'Longitude', 'Fuel Oil #2 Use (kBtu)', "Natural Gas Use (kBtu)", "Fuel Oil #4 Use (kBtu)", "Direct GHG Emissions (Metric Tons CO2e)", "Property GFA - Calculated (Buildings) (ft²)",  "Site EUI (kBtu/ft²)"]]
boiler_df = pd.read_csv("Datas/boiler_permits.csv", thousands=',') # Some volumes contain commas
building_permit_df = pd.read_csv("Datas/DOB_building_permits.csv")[['BOROUGH', 'Issuance Date', 'Expiration Date','LATITUDE', 'LONGITUDE', 'Bin #', 'Lot']]
traffic_df = pd.read_csv("Datas/traffic.csv", thousands=',')
tree_df = pd.read_csv('Datas/tree_data.csv')[['address', 'borough', 'latitude', 'longitude', 'bin']]

# Our sensors: We gotta standardize these lil goonies.
epa_df = pd.read_csv("Datas/epa_output.csv")
openaq_df = pd.read_csv("Datas/openaq_top5_highest_pm25_by_month.csv")
purpleair_df = pd.read_csv("Datas/purple_air_monthlypm25.csv")

/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_12194/3705200772.py:1: DtypeWarning: Columns (10: Postal Code, 16: Largest Property Use Type - Gross Floor Area (ft²), 32: Eligible for Certification for Report PED (Y/N), 34: ENERGY STAR Certification - Application Status, 42: Percent Electricity, 61: Natural Gas Use (therms), 76: Total (Location-Based) GHG Emissions (Metric Tons CO2e), 77: Total (Location-Based) GHG Emissions Intensity (kgCO2e/ft²), 78: Direct GHG Emissions (Metric Tons CO2e), 79: Direct GHG Emissions Intensity (kgCO2e/ft²), 80: Indirect (Location-Based) GHG Emissions (Metric Tons CO2e), 81: Indirect (Location-Based) GHG Emissions Intensity (kgCO2e/ft²), 82: Net Emissions (Metric Tons CO2e), 91: Default Values, 174: Institutional Property? (Y/N), 239: Data Quality Checker Run?, 240: Data Quality Checker - Date Run, 241: Last Modified Date - Property, 242: Last Modified Date - Electric Meters, 243: Electric Distribution Utility, 244: Last Modified Date - Gas M

In [23]:
# Pre-Processing

# Change traffic_df so that it includes traffic counts based on Year, Month, and Latitude Longitude (location)
traffic_df['Vol'] = pd.to_numeric(traffic_df['Vol'])
traffic_df = (
    traffic_df.groupby(["SegmentID", "Yr", "M"])
    .agg(
        count=("Vol", "sum"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first")
    )
    .reset_index()
)


# Change boiler issue and expiration date to dates
boiler_df['ExpirationDate_M'] = pd.to_datetime(boiler_df['ExpirationDate']).dt.month
boiler_df['ExpirationDate_Y'] = pd.to_datetime(boiler_df['ExpirationDate']).dt.year
boiler_df['IssueDate_M'] = pd.to_datetime(boiler_df['IssueDate']).dt.month
boiler_df['IssueDate_Y'] = pd.to_datetime(boiler_df['IssueDate']).dt.year

boiler_df["issue_ym"] = (
    boiler_df["IssueDate_Y"] * 12
    + boiler_df["IssueDate_M"]
)

boiler_df["expiration_ym"] = (
    boiler_df["ExpirationDate_Y"] * 12
    + boiler_df["ExpirationDate_M"]
)


# Do the same for building permits
building_permit_df["Expiration Date"] = pd.to_datetime(
    building_permit_df["Expiration Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

building_permit_df["Issuance Date"] = pd.to_datetime(
    building_permit_df["Issuance Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

building_permit_df["ExpirationDate_M"] = building_permit_df["Expiration Date"].dt.month
building_permit_df["ExpirationDate_Y"] = building_permit_df["Expiration Date"].dt.year

building_permit_df["IssueDate_M"] = building_permit_df["Issuance Date"].dt.month
building_permit_df["IssueDate_Y"] = building_permit_df["Issuance Date"].dt.year

building_permit_df["issue_ym"] = (
    building_permit_df["IssueDate_Y"] * 12
    + building_permit_df["IssueDate_M"]
)

building_permit_df["expiration_ym"] = (
    building_permit_df["ExpirationDate_Y"] * 12
    + building_permit_df["ExpirationDate_M"]
)


In [24]:
# print(len(building_permit_df['Expiration Date']))
# len(building_permit_df[~ building_permit_df['Expiration Date'].isna()])

In [25]:
building_data_gdf = to_gdf(building_data_df, "Latitude", "Longitude")
boiler_gdf = to_gdf(boiler_df, "latitude", "longitude")
building_permit_gdf = to_gdf(building_permit_df, "LATITUDE", "LONGITUDE")
traffic_gdf = to_gdf(traffic_df, 'latitude', 'longitude')
tree_gdf = to_gdf(tree_df, "latitude", "longitude")

In [26]:
tree_gdf.crs

<Projected CRS: EPSG:32618>
Name: WGS 84 / UTM zone 18N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 78°W and 72°W, northern hemisphere between equator and 84°N, onshore and offshore. Bahamas. Canada - Nunavut; Ontario; Quebec. Colombia. Cuba. Ecuador. Greenland. Haiti. Jamaica. Panama. Turks and Caicos Islands. United States (USA). Venezuela.
- bounds: (-78.0, 0.0, -72.0, 84.0)
Coordinate Operation:
- name: UTM zone 18N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [27]:
epa_df = standardize_sensor_data(epa_df, "EPA", epa_col_map)
openaq_df = standardize_sensor_data(openaq_df, "OPENAQ", openaq_col_map)
purpleair_df = standardize_sensor_data(purpleair_df, "PURPLEAIR", purple_col_map)

epa_gdf = to_gdf(epa_df, "latitude", "longitude")
openaq_gdf = to_gdf(openaq_df, "latitude", "longitude")
purpleair_gdf = to_gdf(purpleair_df, "latitude", "longitude")

sensors_gdf = pd.concat([epa_gdf, openaq_gdf, purpleair_gdf], ignore_index=True)

sensors_gdf["sensor_ym"] = (
    sensors_gdf["year"] * 12
    + sensors_gdf["month"]
)

sensors_gdf.head(10)




,source,sensor_id,latitude,longitude,year,month,pm25,sensor_key,geometry,sensor_ym
0,EPA,110,40.8160,-73.90200,2016,1,6.624461,EPA_110,POINT (592600.643 4518911.64),24193
1,EPA,133,40.8679,-73.87809,2016,1,8.990909,EPA_133,POINT (594543.361 4524698.573),24193
2,EPA,110,40.8160,-73.90200,2017,1,6.758297,EPA_110,POINT (592600.643 4518911.64),24205
3,EPA,133,40.8679,-73.87809,2017,1,8.790909,EPA_133,POINT (594543.361 4524698.573),24205
4,EPA,110,40.8160,-73.90200,2018,1,7.876813,EPA_110,POINT (592600.643 4518911.64),24217
5,EPA,133,40.8679,-73.87809,2018,1,11.110000,EPA_133,POINT (594543.361 4524698.573),24217
6,EPA,110,40.8160,-73.90200,2019,1,7.226107,EPA_110,POINT (592600.643 4518911.64),24229
7,EPA,133,40.8679,-73.87809,2019,1,7.420000,EPA_133,POINT (594543.361 4524698.573),24229
8,EPA,110,40.8160,-73.90200,2020,1,8.254894,EPA_110,POINT (592600.643 4518911.64),24241
9,EPA,133,40.8679,-73.87809,2020,1,7.245455,EPA_133,POINT (594543.361 4524698.573),24241


In [ ]:
# Time to Aggregate 

# parameters for function
    # sensors_gdf,
    # features_gdf,
    # prefix,
    # radius=1609.344,
    # mode="monthly",
    # year_col=None,
    # month_col=None,
    # agg_cols=None,
    # agg_type="sum"


# boiler_gdf.columns

boiler_agg = aggregate_near_sensors(
    sensors_gdf,
    boiler_gdf,
    prefix="boiler",
    mode="interval",
    agg_type="count"
)

traffic_agg = aggregate_near_sensors(
    sensors_gdf,
    traffic_gdf,
    prefix='traffic',
    year_col='Yr',
    month_col='M',
    mode='monthly',
    agg_cols={'count' : 'sum'}
)


building_cols = {
      "Fuel Oil #2 Use (kBtu)" : "sum",
      "Fuel Oil #4 Use (kBtu)" : "sum",
      "Natural Gas Use (kBtu)" :  "sum",
      "Direct GHG Emissions (Metric Tons CO2e)" :  "sum",
      "Property GFA - Calculated (Buildings) (ft²)" :  "sum",
      "Site EUI (kBtu/ft²)" : "mean"
}
# Convert columns to numeric
for col in building_cols:
    building_data_gdf[col] = pd.to_numeric(
        building_data_gdf[col]
        .astype("string")
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

building_data_gdf['Calendar Year'] = pd.to_numeric(building_data_gdf['Calendar Year'])

building_data_agg = aggregate_near_sensors(
    sensors_gdf,
    building_data_gdf,
    prefix='buildings',
    year_col='Calendar Year',
    mode='yearly',
    agg_cols=building_cols)

building_permit_agg = aggregate_near_sensors(
    sensors_gdf,
    building_permit_gdf,
    prefix='building_permit',
    mode='interval',
    agg_type='count'
)

tree_agg = aggregate_near_sensors(
    sensors_gdf,
    tree_gdf,
    prefix='tree',
    mode='static',
    agg_type='count'
)

# final_df = (
#     sensors_gdf
#     .merge(traffic_agg, on="sensor_id", how="left")
#     .merge(boiler_agg, on="sensor_id", how="left")
# )



In [39]:
final_df = (
    sensors_gdf
    .merge(traffic_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(boiler_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(building_data_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(building_permit_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(tree_agg, on=["sensor_key", "year", "month"], how="left")
)

final_df.head(20)
final_df.to_csv('final_data.csv', index=False)


# final_df["sensor_key"].unique()
# final_df.loc[final_df["tree_count"].isna(), "sensor_key"].unique()


